# C07 — Vision Transformers: ViT, DeiT, and Swin

> **Audience**: PhD students · **Framework**: PyTorch + Hugging Face · **Dataset**: MNIST / Beans

Vision Transformers (ViT) replaced convolutional inductive biases with
pure self-attention, achieving state-of-the-art on ImageNet with enough data
or the right training recipe.

**What this notebook builds**
1. Patch embedding — converting an image into a token sequence (from scratch)
2. Multi-head self-attention — the core ViT operation (from scratch)
3. Full ViT encoder — complete transformer stack (from scratch)
4. Pretrained ViT fine-tuning via Hugging Face
5. Swin Transformer — hierarchical ViT for dense prediction
6. DeiT — data-efficient ViT via knowledge distillation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import math

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
print(f"Device: {DEVICE}")

# 1) Patch Embedding

**The core idea of ViT** (Dosovitskiy et al., 2020)

Images are split into non-overlapping patches, each flattened and projected
to an embedding dimension. This converts an image into a *sequence of tokens*
that a standard Transformer encoder can process.

**Why this matters**
- No spatial inductive bias: attention attends to *all* patches equally,
  regardless of distance. CNNs can only attend within the kernel window.
- Scales well with data: more data → attention learns better long-range
  dependencies; CNNs plateau faster.
- Weakness: requires more data than CNNs to learn spatial patterns from scratch
  (addressed by DeiT in Section 6).

**Sample I/O**
```
Input image    : (batch_num, n_channels=1, H=28, W=28)
Patch size p=4 : grid of 7×7 = 49 patches, each of size 4×4×1 = 16 features
After linear   : (batch_num, num_patches=49, embed_dim)
After CLS token: (batch_num, num_patches+1=50, embed_dim)
```

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Splits an image into patches and linearly embeds each patch.

    This is equivalent to a Conv2d with kernel_size=patch_size and
    stride=patch_size — a single non-overlapping convolution.

    Args:
        img_size    : Input image height/width (assumed square)
        patch_size  : Size of each square patch
        in_channels : Number of input channels
        embed_dim   : Embedding dimension for each patch token
    """

    def __init__(
        self,
        img_size: int   = 28,
        patch_size: int = 4,
        in_channels: int = 1,
        embed_dim: int  = 128,
    ) -> None:
        super().__init__()

        assert img_size % patch_size == 0, "img_size must be divisible by patch_size"

        self.num_patches = (img_size // patch_size) ** 2
        self.patch_size  = patch_size

        # 1 non-overlapping conv == split into patches + linear projection
        # (batch_num, in_channels, H, W) → (batch_num, embed_dim, H//p, W//p)
        self.proj = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (batch_num, in_channels, H, W)

        Returns:
            patches : (batch_num, num_patches, embed_dim)
        """
        # (batch_num, in_channels, H, W) → (batch_num, embed_dim, H//p, W//p)
        x = self.proj(x)
        # (batch_num, embed_dim, H//p, W//p) → (batch_num, embed_dim, num_patches)
        x = x.flatten(2)
        # (batch_num, embed_dim, num_patches) → (batch_num, num_patches, embed_dim)
        return x.transpose(1, 2)


# ── Verify ─────────────────────────────────────────────────────────────────────
pe = PatchEmbedding(img_size=28, patch_size=4, in_channels=1, embed_dim=128)
x_img = torch.zeros(4, 1, 28, 28)
with torch.no_grad():
    patches = pe(x_img)
print(f"Image    : {x_img.shape}")   # (4, 1, 28, 28)
print(f"Patches  : {patches.shape}") # (4, 49, 128)  ← 49 = 7×7 patches

# 2) Multi-Head Self-Attention

**Self-attention intuition**
Each token (patch) attends to every other token and weights their value vectors
by how relevant they are to a learned query. The output is a weighted mixture
of all value vectors.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Multi-head** splits the embedding into `num_heads` sub-spaces, runs attention
in each independently, then concatenates. Different heads can learn different
relationships (e.g., spatial proximity, semantic similarity, texture vs shape).

**$\sqrt{d_k}$ scaling**
Without scaling, large dot products push softmax into saturation, producing
near-zero gradients. Dividing by $\sqrt{d_k}$ keeps the pre-softmax logits
in a reasonable range regardless of embedding dimension.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """
    Standard multi-head self-attention module.

    All three projection matrices (Q, K, V) are combined into one linear layer
    for efficiency, then split after projection.

    Args:
        embed_dim : Token embedding dimension (model_dim)
        num_heads : Number of parallel attention heads
        dropout   : Attention weight dropout probability
    """

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"

        self.embed_dim  = embed_dim
        self.num_heads  = num_heads
        self.head_dim   = embed_dim // num_heads  # dimension per head
        self.scale      = self.head_dim ** -0.5   # 1 / sqrt(d_k)

        # Fused QKV projection: one matrix for all three projections
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, 3*embed_dim)
        self.qkv     = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (batch_num, seq_len, embed_dim)

        Returns:
            out : (batch_num, seq_len, embed_dim)
        """
        batch_num, seq_len, _ = x.shape

        # Project to Q, K, V simultaneously
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, 3*embed_dim)
        qkv = self.qkv(x)

        # Split and reshape to (batch_num, num_heads, seq_len, head_dim)
        qkv = qkv.view(batch_num, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, batch_num, num_heads, seq_len, head_dim)
        Q, K, V = qkv.unbind(0)            # each (batch_num, num_heads, seq_len, head_dim)

        # Scaled dot-product attention
        # (batch_num, num_heads, seq_len, head_dim) × (batch_num, num_heads, head_dim, seq_len)
        # → (batch_num, num_heads, seq_len, seq_len)
        attn_weights = (Q @ K.transpose(-2, -1)) * self.scale
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)

        # Weighted sum over value vectors
        # (batch_num, num_heads, seq_len, seq_len) × (batch_num, num_heads, seq_len, head_dim)
        # → (batch_num, num_heads, seq_len, head_dim)
        out = attn_weights @ V

        # Merge heads back into embed_dim
        # (batch_num, num_heads, seq_len, head_dim) → (batch_num, seq_len, embed_dim)
        out = out.transpose(1, 2).contiguous().view(batch_num, seq_len, self.embed_dim)

        # Final linear projection
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        return self.out_proj(out)


# ── Verify ─────────────────────────────────────────────────────────────────────
mhsa = MultiHeadSelfAttention(embed_dim=128, num_heads=4)
x_attn = torch.zeros(2, 50, 128)  # (batch_num, seq_len, embed_dim)
with torch.no_grad():
    out_attn = mhsa(x_attn)
print(f"MHSA  in : {x_attn.shape}")   # (2, 50, 128)
print(f"MHSA out : {out_attn.shape}") # (2, 50, 128)  ← same shape

# 3) ViT Encoder Block and Full ViT

**Transformer encoder block**
```
LayerNorm → MHSA → Residual (+x)
LayerNorm → MLP  → Residual (+x)
```

**Pre-norm vs post-norm**
The original Transformer (Vaswani et al.) uses post-norm (LN after residual).
ViT uses *pre-norm* (LN before attention). Pre-norm is more stable for deep
networks and is now the standard in vision transformers.

**Classification token (CLS)**
A learnable CLS token is prepended to the patch sequence.
After all encoder blocks, only the CLS token is passed to the classification head.
The CLS token aggregates information from all patches via attention.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """
    One ViT encoder block: pre-norm MHSA + pre-norm MLP, both with residual.
    """

    def __init__(self, embed_dim: int, num_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.1) -> None:
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = MultiHeadSelfAttention(embed_dim, num_heads, dropout)

        self.norm2 = nn.LayerNorm(embed_dim)
        # MLP expands to mlp_ratio × embed_dim then contracts back
        hidden_dim = int(embed_dim * mlp_ratio)
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        self.mlp   = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),  # GELU preferred over ReLU in transformers
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-norm MHSA with residual
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        x = x + self.attn(self.norm1(x))
        # Pre-norm MLP with residual
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        x = x + self.mlp(self.norm2(x))
        return x


class ViT(nn.Module):
    """
    Vision Transformer for image classification.

    Architecture:
        Patch embedding → [CLS] token → Positional embedding
        → num_layers × TransformerEncoderBlock → LayerNorm
        → CLS token → classification head
    """

    def __init__(
        self,
        img_size:    int = 28,
        patch_size:  int = 4,
        in_channels: int = 1,
        num_classes: int = 10,
        embed_dim:   int = 128,
        num_layers:  int = 6,
        num_heads:   int = 4,
        mlp_ratio:   float = 4.0,
        dropout:     float = 0.1,
    ) -> None:
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches      = self.patch_embed.num_patches

        # Learnable CLS token — one vector prepended to the patch sequence
        # (1, 1, embed_dim) — broadcast over batch
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Learnable positional embeddings — one per patch + one for CLS
        # (1, num_patches+1, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

        self.pos_drop = nn.Dropout(dropout)

        # Stack of transformer encoder blocks
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        # Classification head: only uses the CLS token output
        # (batch_num, embed_dim) → (batch_num, num_classes)
        self.head = nn.Linear(embed_dim, num_classes)

        # Initialise weights — important for transformer stability
        self._init_weights()

    def _init_weights(self) -> None:
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_num = x.size(0)

        # Convert image to patch token sequence
        # (batch_num, in_channels, H, W) → (batch_num, num_patches, embed_dim)
        x = self.patch_embed(x)

        # Expand and prepend CLS token
        # (1, 1, embed_dim) → (batch_num, 1, embed_dim)
        cls = self.cls_token.expand(batch_num, -1, -1)
        # (batch_num, num_patches+1, embed_dim)
        x = torch.cat([cls, x], dim=1)

        # Add learned positional embeddings
        # (batch_num, num_patches+1, embed_dim)
        x = self.pos_drop(x + self.pos_embed)

        # Apply transformer encoder blocks
        for block in self.blocks:
            # (batch_num, num_patches+1, embed_dim) → same
            x = block(x)

        x = self.norm(x)

        # Extract CLS token and classify
        # (batch_num, embed_dim)
        cls_out = x[:, 0]
        # (batch_num, num_classes)
        return self.head(cls_out)


# ── Shape dry-run ──────────────────────────────────────────────────────────────
vit = ViT(img_size=28, patch_size=4, in_channels=1, num_classes=10,
           embed_dim=128, num_layers=6, num_heads=4)
x_vit = torch.zeros(4, 1, 28, 28)
with torch.no_grad():
    out_vit = vit(x_vit)

print(f"ViT input  : {x_vit.shape}")   # (4, 1, 28, 28)
print(f"ViT output : {out_vit.shape}") # (4, 10)
print(f"Parameters : {sum(p.numel() for p in vit.parameters()):,}")

# 4) Train ViT on MNIST

**Important observation for research**
ViT needs significantly more data than CNNs to learn spatial inductive biases
from scratch. On MNIST (60 000 images, 28×28) it reaches good accuracy, but
would be clearly outperformed by a CNN of similar parameter count on a harder
dataset without ImageNet pretraining.

This is why Section 5 (pretrained ViT) and Section 6 (DeiT) exist.

In [ ]:
# Data loading for MNIST
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
mnist_tr = torchvision.datasets.MNIST("./data", True,  download=True, transform=transform)
mnist_vl = torchvision.datasets.MNIST("./data", False, download=True, transform=transform)
tr_loader = DataLoader(mnist_tr, batch_size=128, shuffle=True,  num_workers=2)
vl_loader = DataLoader(mnist_vl, batch_size=256, shuffle=False, num_workers=2)


def train_vit(model, tr_loader, vl_loader, n_epochs=5, lr=3e-4):
    """Standard training loop with AdamW + cosine schedule."""
    model.to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, n_epochs + 1):
        model.train()
        running_loss = 0.0
        for X, y in tr_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            # (batch_num, 1, 28, 28) → (batch_num, 10)
            loss = criterion(model(X), y)
            loss.backward(); optimizer.step()
            running_loss += loss.item() * X.size(0)
        scheduler.step()

        model.eval()
        correct = 0
        with torch.no_grad():
            for X, y in vl_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                correct += (model(X).argmax(1) == y).sum().item()

        tl  = running_loss / len(tr_loader.dataset)
        acc = correct / len(vl_loader.dataset)
        print(f"Epoch {epoch}/{n_epochs} | loss={tl:.4f} | val_acc={acc:.3f}")


torch.manual_seed(0)
vit_model = ViT(img_size=28, patch_size=4, in_channels=1, num_classes=10,
                embed_dim=128, num_layers=6, num_heads=4, dropout=0.1)
train_vit(vit_model, tr_loader, vl_loader, n_epochs=5, lr=3e-4)

# 4.5) Dynamic Resolution — Interpolating Positional Embeddings

**The problem with fixed positional embeddings**

Our `ViT` stores a `pos_embed` tensor of shape `(1, num_patches + 1, embed_dim)`,
where `num_patches = (img_size // patch_size) ** 2`.
If the input image has a different size at inference time, the number of patches
changes — and the stored positional embeddings no longer have the right shape.

**Why this matters for research**
- Pretrained on 224×224 but want to evaluate on 384×384 (higher res, better accuracy)
- Multi-scale inference for dense prediction tasks (segmentation, detection)
- Fine-tuning on datasets with different aspect ratios

**The fix: bicubic interpolation of positional embeddings**

The patch positional embeddings encode a 2D spatial grid.
We can treat them as a low-resolution 2D feature map and **bicubically upsample**
them to any target resolution — the same operation used to resize images.

```
pos_embed (excl. CLS) :  (1, old_grid_h, old_grid_w, embed_dim)
                              ↓  F.interpolate bicubic
                         (1, new_grid_h, new_grid_w, embed_dim)
```

This is how the original ViT paper, DINOv2, and all Hugging Face ViT models
handle variable-resolution inference. The HF `ViTImageProcessor` does it
automatically via `interpolate_pos_encodings=True`.

In [ ]:
def interpolate_pos_embed(
    pos_embed: torch.Tensor,
    old_grid_size: int,
    new_grid_size: int,
    embed_dim: int,
) -> torch.Tensor:
    """
    Bicubically interpolates patch positional embeddings to a new grid resolution.

    The CLS token embedding is left unchanged; only the patch tokens are resized.
    Uses bicubic interpolation (same as image resizing) because positional
    embeddings are smooth 2D functions of spatial position.

    Args:
        pos_embed     : (1, old_num_patches + 1, embed_dim)  — full stored embedding
        old_grid_size : sqrt(old_num_patches), e.g. 7 for a 28×28 image with patch=4
        new_grid_size : Target grid size, e.g. 14 for a 56×56 image with patch=4
        embed_dim     : Embedding dimension

    Returns:
        pos_embed_interp : (1, new_num_patches + 1, embed_dim)  — interpolated embedding
    """
    if old_grid_size == new_grid_size:
        return pos_embed  # no interpolation needed

    # Separate CLS token (first token) from patch tokens
    # (1, 1, embed_dim)
    cls_token_embed   = pos_embed[:, :1, :]
    # (1, old_num_patches, embed_dim)
    patch_embed       = pos_embed[:, 1:, :]

    # Reshape patch tokens to a 2D spatial grid for interpolation
    # (1, old_num_patches, embed_dim) → (1, old_grid_size, old_grid_size, embed_dim)
    patch_embed = patch_embed.view(1, old_grid_size, old_grid_size, embed_dim)

    # Rearrange to (1, embed_dim, old_grid_size, old_grid_size) for F.interpolate
    patch_embed = patch_embed.permute(0, 3, 1, 2)

    # Bicubic upsample to new spatial resolution
    # (1, embed_dim, old_grid_size, old_grid_size) → (1, embed_dim, new_grid_size, new_grid_size)
    patch_embed = F.interpolate(
        patch_embed.float(),
        size=(new_grid_size, new_grid_size),
        mode="bicubic",
        align_corners=False,
    ).to(pos_embed.dtype)

    # Rearrange back to (1, new_num_patches, embed_dim)
    # (1, embed_dim, new_grid_size, new_grid_size) → (1, new_grid_size^2, embed_dim)
    patch_embed = patch_embed.permute(0, 2, 3, 1)
    patch_embed = patch_embed.view(1, new_grid_size * new_grid_size, embed_dim)

    # Concatenate CLS token back at position 0
    # (1, new_num_patches + 1, embed_dim)
    return torch.cat([cls_token_embed, patch_embed], dim=1)


class ViTDynamic(ViT):
    """
    ViT with dynamic resolution support via positional embedding interpolation.

    Inherits the full ViT architecture; overrides only the forward method to
    detect resolution changes and interpolate pos_embed accordingly.

    This is identical to how ViT-Base/16 handles resolution changes in the
    official JAX implementation and all Hugging Face ViT checkpoints.
    """

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_num, _, H, W = x.shape

        # Compute number of patches in this forward pass
        new_grid_h = H // self.patch_embed.patch_size
        new_grid_w = W // self.patch_embed.patch_size
        new_num_patches = new_grid_h * new_grid_w

        # Training resolution grid size (stored at init)
        old_num_patches = self.patch_embed.num_patches
        old_grid_size   = int(old_num_patches ** 0.5)

        # Convert image to patch tokens
        # (batch_num, in_channels, H, W) → (batch_num, new_num_patches, embed_dim)
        x = self.patch_embed(x)

        # Prepend CLS token
        # (batch_num, new_num_patches + 1, embed_dim)
        cls = self.cls_token.expand(batch_num, -1, -1)
        x   = torch.cat([cls, x], dim=1)

        # Interpolate positional embeddings if resolution has changed
        # (1, old_num_patches+1, embed_dim) → (1, new_num_patches+1, embed_dim)
        pos_embed = interpolate_pos_embed(
            self.pos_embed,
            old_grid_size = old_grid_size,
            new_grid_size = new_grid_h,  # assumes square grid
            embed_dim     = self.pos_embed.shape[-1],
        )

        # Add (possibly interpolated) positional embeddings
        # (batch_num, new_num_patches+1, embed_dim)
        x = self.pos_drop(x + pos_embed)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        # Use CLS token for classification
        # (batch_num, num_classes)
        return self.head(x[:, 0])


# ── Demonstration: same model weights, three different input resolutions ───────
torch.manual_seed(0)
vit_dyn = ViTDynamic(
    img_size=28, patch_size=4, in_channels=1, num_classes=10,
    embed_dim=128, num_layers=6, num_heads=4,
)
# Load the weights we trained in Section 4 (same architecture)
vit_dyn.load_state_dict(vit_model.state_dict())
vit_dyn.eval()

print("Resolution → grid size → pos_embed shape → output shape")
print("-" * 60)
for res in [28, 56, 112]:
    x_test = torch.zeros(2, 1, res, res)
    grid   = res // 4
    with torch.no_grad():
        out = vit_dyn(x_test)
    expected_pos_dim = grid * grid + 1
    print(f"  {res}×{res}  →  {grid}×{grid}={grid**2} patches  →  "
          f"pos_embed interpolated to ({1}, {expected_pos_dim}, 128)  →  {out.shape}")

## 4.6 Non-Square and Rectangular Inputs

Standard ViT assumes square images and a square patch grid.
For rectangular inputs (common in medical imaging, satellite imagery, documents),
we need to interpolate with `new_grid_h ≠ new_grid_w`.

The `interpolate_pos_embed` function above already handles this because
`F.interpolate` with `size=(h, w)` supports non-square targets.
The only adjustment needed is tracking `grid_h` and `grid_w` separately.

In [ ]:
class ViTDynamicRect(ViTDynamic):
    """
    ViT with full non-square resolution support.
    Tracks grid_h and grid_w separately to handle rectangular inputs.
    """

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_num, _, H, W = x.shape
        p             = self.patch_embed.patch_size
        new_grid_h    = H // p
        new_grid_w    = W // p
        new_num_patches = new_grid_h * new_grid_w
        old_grid_size = int(self.patch_embed.num_patches ** 0.5)

        # (batch_num, in_channels, H, W) → (batch_num, new_num_patches, embed_dim)
        x = self.patch_embed(x)
        cls = self.cls_token.expand(batch_num, -1, -1)
        x   = torch.cat([cls, x], dim=1)

        # Rectangular interpolation: old square grid → new (h, w) grid
        cls_emb   = self.pos_embed[:, :1, :]
        patch_emb = self.pos_embed[:, 1:, :]
        embed_dim = patch_emb.shape[-1]

        # (1, old_num_patches, embed_dim) → (1, embed_dim, old_g, old_g)
        patch_emb = patch_emb.view(1, old_grid_size, old_grid_size, embed_dim)
        patch_emb = patch_emb.permute(0, 3, 1, 2)

        # (1, embed_dim, old_g, old_g) → (1, embed_dim, new_grid_h, new_grid_w)
        patch_emb = F.interpolate(
            patch_emb.float(),
            size=(new_grid_h, new_grid_w),
            mode="bicubic",
            align_corners=False,
        ).to(self.pos_embed.dtype)

        # (1, embed_dim, new_grid_h, new_grid_w) → (1, new_num_patches, embed_dim)
        patch_emb = patch_emb.permute(0, 2, 3, 1).view(1, new_num_patches, embed_dim)
        pos_embed = torch.cat([cls_emb, patch_emb], dim=1)

        x = self.pos_drop(x + pos_embed)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])


# ── Non-square demo ────────────────────────────────────────────────────────────
vit_rect = ViTDynamicRect(
    img_size=28, patch_size=4, in_channels=1, num_classes=10,
    embed_dim=128, num_layers=6, num_heads=4,
)
vit_rect.load_state_dict(vit_model.state_dict())
vit_rect.eval()

print("Non-square inputs:")
for (H, W) in [(28, 28), (28, 56), (56, 112), (112, 56)]:
    x_rect = torch.zeros(2, 1, H, W)
    grid_h, grid_w = H // 4, W // 4
    with torch.no_grad():
        out_rect = vit_rect(x_rect)
    print(f"  Input {H}×{W}  →  grid {grid_h}×{grid_w}={grid_h*grid_w} patches  →  out {out_rect.shape}")


# ── Visualise positional embedding interpolation ───────────────────────────────
# Show how each patch position's embedding changes across resolutions
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
resolutions = [28, 56, 112]
for ax, res in zip(axes, resolutions):
    grid = res // 4
    old_g = 7  # trained on 28×28 with patch=4

    cls_e  = vit_dyn.pos_embed[:, :1, :]
    patch_e = vit_dyn.pos_embed[:, 1:, :]
    dim    = patch_e.shape[-1]

    patch_e = patch_e.view(1, old_g, old_g, dim).permute(0, 3, 1, 2)
    patch_e = F.interpolate(patch_e.float(), size=(grid, grid),
                            mode="bicubic", align_corners=False)
    patch_e = patch_e.permute(0, 2, 3, 1).view(grid, grid, dim)

    # Show norm of each patch position embedding as a proxy for its strength
    # (grid, grid)
    norms = patch_e.norm(dim=-1).detach().numpy()
    im = ax.imshow(norms, cmap="viridis")
    ax.set_title(f"Res {res}×{res}  →  {grid}×{grid} grid")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("Positional embedding norms after bicubic interpolation")
plt.tight_layout(); plt.show()

# 5) Pretrained ViT Fine-Tuning (Hugging Face)

Using a pretrained ViT-Base/16 (85M parameters, pretrained on ImageNet-21K)
and fine-tuning it on a small domain-specific dataset.

**ViT-Base/16**: 16×16 patch size, 197 tokens (196 patches + CLS), 12 layers,
768 hidden dim, 12 heads.

**Fine-tuning strategy**: load pretrained weights, freeze all layers except
the classification head for Phase 1, then unfreeze all for Phase 2 (identical
to the pattern in c03).

In [ ]:
from transformers import ViTForImageClassification, ViTImageProcessor
from datasets import load_dataset

# ── Load Beans disease dataset (small, 3 classes) ─────────────────────────────
beans_ds = load_dataset("beans")
labels   = beans_ds["train"].features["labels"].names
print(f"Classes: {labels}")
print(f"Train: {len(beans_ds['train'])} | Val: {len(beans_ds['validation'])}")

processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")

vit_pretrained = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=len(labels),
    id2label={str(i): c for i, c in enumerate(labels)},
    label2id={c: str(i) for i, c in enumerate(labels)},
    ignore_mismatched_sizes=True,  # head has new size
)

# Count and show frozen/unfrozen split
total     = sum(p.numel() for p in vit_pretrained.parameters())
trainable = sum(p.numel() for p in vit_pretrained.classifier.parameters())
print(f"\nTotal params     : {total:,}")
print(f"Trainable (head) : {trainable:,}  ({100*trainable/total:.2f}%)")

# Quick inference to verify shapes
sample_img  = beans_ds["train"][0]["image"]
inputs      = processor(images=sample_img, return_tensors="pt")
# inputs["pixel_values"]: (1, 3, 224, 224)

with torch.no_grad():
    outputs = vit_pretrained(**inputs)
    # logits: (1, num_labels)
    pred = outputs.logits.argmax(-1).item()

print(f"\nSample prediction: {labels[pred]}")
print(f"True label       : {labels[beans_ds['train'][0]['labels']]}")

# 6) DeiT and Swin Transformer

## 6.1 DeiT — Data-Efficient Image Transformers

**The problem DeiT solves**
Original ViT requires 300M+ images (JFT-300M) to outperform CNNs.
DeiT (Touvron et al., 2021) achieves competitive results using only ImageNet
(1.2M images) through:

1. **Aggressive augmentation**: CutMix, MixUp, RandAugment
2. **Knowledge distillation**: a teacher CNN (e.g., RegNetY-16GF) supervises
   the ViT student via a special *distillation token*

**Distillation token**
DeiT adds a second learnable token alongside CLS:
- CLS token: trained with standard cross-entropy against true label
- Distillation token: trained with hard/soft label from teacher model
- At inference: average the two token predictions

## 6.2 Swin Transformer — Hierarchical ViT

**Why hierarchical matters**
Original ViT uses fixed-size patches with global attention — quadratic in number
of tokens, and no multi-scale features (needed for detection/segmentation).

Swin (Liu et al., 2021) introduces:
1. **Window attention**: attention within local 7×7 windows → linear complexity
2. **Shifted windows**: windows shift each layer to enable cross-window connections
3. **Hierarchical stages**: like ResNet, spatial resolution decreases and channels
   increase through the network — enabling FPN-style multi-scale feature maps

Swin-T is the standard backbone for many detection/segmentation papers.

In [ ]:
from transformers import AutoFeatureExtractor, SwinForImageClassification

# ── Load pretrained Swin-Tiny ─────────────────────────────────────────────────
swin_processor = AutoFeatureExtractor.from_pretrained("microsoft/swin-tiny-patch4-window7-224")
swin_model     = SwinForImageClassification.from_pretrained(
    "microsoft/swin-tiny-patch4-window7-224"
).to(DEVICE)

print("Swin-Tiny configuration:")
print(f"  Parameters   : {sum(p.numel() for p in swin_model.parameters()):,}")
print(f"  Patch size   : {swin_model.config.patch_size}")
print(f"  Window size  : {swin_model.config.window_size}")
print(f"  Depths       : {swin_model.config.depths}  (blocks per stage)")
print(f"  Num heads    : {swin_model.config.num_heads}")
print(f"  ImageNet-1K classes: {swin_model.config.num_labels}")

# Sample inference
sample_img   = beans_ds["train"][5]["image"]
inputs_swin  = swin_processor(images=sample_img, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs_swin = swin_model(**inputs_swin)
    # logits: (1, 1000)  — ImageNet-1K classes
    top5_idx     = outputs_swin.logits[0].topk(5).indices.tolist()
    imagenet_labels = swin_model.config.id2label

print("\nSwin-Tiny top-5 ImageNet predictions on a Beans image:")
for idx in top5_idx:
    print(f"  {imagenet_labels[idx]}")

# ── Model comparison table ─────────────────────────────────────────────────────
print("\n" + "="*60)
print(f"{'Model':<20} {'Params':>10} {'Attn type':>18} {'Scales':>8}")
print("="*60)
models_info = [
    ("ViT-B/16",   "86M",  "Global (full)",       "Single"),
    ("DeiT-B/16",  "86M",  "Global + distill",    "Single"),
    ("Swin-T",     "28M",  "Local window+shift",  "Multi"),
    ("Swin-B",     "88M",  "Local window+shift",  "Multi"),
]
for name, params, attn, scales in models_info:
    print(f"{name:<20} {params:>10} {attn:>18} {scales:>8}")

# Summary

| Model | Attention | Data needed | Inductive bias | Best use |
|---|---|---|---|---|
| ViT-B/16 | Global | Large (JFT/IN-21K) | None | General classification with pretraining |
| DeiT-B | Global + distil. | ImageNet only | From teacher | Data-efficient classification |
| Swin-T | Local window | ImageNet | Hierarchical | Detection, segmentation backbones |

**CNN vs ViT design philosophy**

| Property | CNN | ViT |
|---|---|---|
| Translation invariance | Built-in | Learned from data |
| Long-range dependencies | Expensive (deep stacks) | Natural (attention) |
| Scales with data | Plateaus | Continues improving |
| Data efficiency | Strong | Weak without pretraining |

**Key research insight**
The choice between ViT and CNN is not absolute — ConvNeXt (Liu et al., 2022)
modernised the CNN design space to match ViT performance, showing that the
training recipe and architecture details matter as much as the attention mechanism.
Modern practice: use Swin or ViT pretrained on ImageNet-21K; fine-tune on your task.

# 7) Modern Dynamic-Resolution ViT (Qwen2.5-VL Architecture)

The interpolation in Section 4.5 rescales *fixed* positional embeddings after the fact.
Qwen2.5-VL takes a fundamentally different approach: **process every image at its
native aspect ratio** with **no resizing to a fixed 224×224 grid**.

A tall document and a wide landscape produce *different numbers of visual tokens*.
The model must handle variable-length token sequences natively.

**Architectural changes vs standard ViT**

| Component | Standard ViT | Qwen2.5-VL |
|---|---|---|
| Input resize | Always 224×224 | Native aspect ratio |
| Positional encoding | Learned absolute | mRoPE (2D/3D rotary) |
| Attention | Full global | Window (local) + periodic global |
| FFN activation | GELU | SwiGLU |
| Normalisation | LayerNorm | RMSNorm |
| Output | Last layer CLS | All-layer fusion (DeepStack) |

Each change is motivated by alignment with the LLM that processes the visual tokens.

## 7.1 Smart Resize — Native Aspect Ratio with Pixel Budget

**The problem with fixed-resolution ViT**
Squashing a 1920×1080 image to 224×224 loses aspect ratio and detail.
Worse, a 4:3 crop receives the same number of tokens as a 16:9 panorama —
despite containing very different amounts of visual information.

**Qwen2.5-VL's algorithm**
1. Compute how many tokens the native resolution would produce:
   $N = \lceil H/p \rceil \times \lceil W/p \rceil$
2. Scale uniformly so $N \in [N_{min}, N_{max}]$ — the *pixel budget*
3. Round H and W independently to the nearest multiple of $p$

This preserves aspect ratio while bounding compute.

In [ ]:
import math

def smart_resize(
    height: int,
    width: int,
    factor: int = 28,           # Qwen uses patch_size=14; factor = 2×14 for 2×2 merge
    min_pixels: int = 4 * 28 * 28,
    max_pixels: int = 16384 * 28 * 28,
) -> tuple:
    """
    Resizes (H, W) to the nearest multiples of `factor` while:
      - preserving aspect ratio
      - keeping H × W within [min_pixels, max_pixels]

    This is a faithful reimplementation of Qwen2.5-VL's smart_resize logic.

    Args:
        height     : Original image height in pixels
        width      : Original image width in pixels
        factor     : Spatial granularity — output dims are multiples of this
        min_pixels : Minimum total pixel count after resize
        max_pixels : Maximum total pixel count after resize (compute budget)

    Returns:
        (new_h, new_w) : Resized dimensions, both divisible by `factor`
    """
    if height < factor or width < factor:
        raise ValueError(f"Image too small for factor={factor}: ({height}, {width})")

    # Compute the scale needed to fit within the pixel budget
    h_bar = round(height / factor) * factor
    w_bar = round(width  / factor) * factor

    # If the rounded size exceeds the budget, shrink proportionally
    if h_bar * w_bar > max_pixels:
        scale = math.sqrt(max_pixels / (h_bar * w_bar))
        h_bar = math.floor(height * scale / factor) * factor
        w_bar = math.floor(width  * scale / factor) * factor

    # If the rounded size is below the minimum, grow proportionally
    elif h_bar * w_bar < min_pixels:
        scale = math.sqrt(min_pixels / (h_bar * w_bar))
        h_bar = math.ceil(height * scale / factor) * factor
        w_bar = math.ceil(width  * scale / factor) * factor

    return h_bar, w_bar


# ── Demonstrate native-aspect-ratio token counts ───────────────────────────────
PATCH = 14   # Qwen2.5-VL patch size
FACTOR = PATCH  # one patch per factor pixels

images_meta = [
    ("Square photo",   224,  224),
    ("Portrait photo",  224,  336),
    ("Landscape photo", 640,  480),
    ("4K widescreen",  3840, 2160),
    ("Tall document",   400, 1200),
    ("Tiny thumbnail",   32,   32),
]

print(f"{'Image':<22} {'Original':>14}  {'After smart_resize':>18}  {'Tokens':>8}")
print("-" * 70)
for name, H, W in images_meta:
    try:
        new_h, new_w = smart_resize(H, W, factor=FACTOR,
                                    min_pixels=4*FACTOR*FACTOR,
                                    max_pixels=4096*FACTOR*FACTOR)
        n_tokens = (new_h // PATCH) * (new_w // PATCH)
        print(f"{name:<22} {H}×{W:>5}  →  {new_h}×{new_w:<6}  {n_tokens:>8} tokens")
    except ValueError as e:
        print(f"{name:<22} {H}×{W:>5}  ERROR: {e}")

## 7.2 mRoPE — Multi-dimensional Rotary Position Embeddings

**Why not learned absolute position embeddings?**

Learned absolute embeddings (our ViT in Section 1) have two problems at variable resolution:
- Fixed size: cannot handle sequences longer than trained on without interpolation
- Non-extrapolating: position 500 is meaningless if training only saw up to 196 patches

**Standard RoPE** (Su et al., 2021) encodes position $p$ as a *rotation* applied to
query and key vectors. Because it uses the *relative* angle difference between tokens,
it extrapolates naturally to unseen positions.

**mRoPE** extends RoPE from 1D (token index) to 3D (time, height, width):

$$\theta_i = \frac{1}{10000^{2i/d}}$$

For each patch at position $(t, h, w)$:
- Dimensions $[0, d/3)$: rotated by $t \cdot \theta_i$  (temporal)
- Dimensions $[d/3, 2d/3)$: rotated by $h \cdot \theta_i$ (height)
- Dimensions $[2d/3, d)$: rotated by $w \cdot \theta_i$ (width)

For static images, $t = 0$ for every patch, making the time component trivial.
For video, $t$ is the frame index, giving each frame a different temporal encoding.

In [ ]:
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """
    Splits the last dimension in half and rotates:
        [x1, x2] → [-x2, x1]

    This is the key operation that makes RoPE work:
    multiplying by cos and adding rotate_half * sin produces a 2D rotation.
    """
    # (... , head_dim) → (... , head_dim)
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def precompute_1d_freqs(
    dim: int, max_len: int = 4096, base: float = 10000.0
) -> tuple:
    """
    Precomputes cos/sin tables for 1D RoPE.

    Args:
        dim     : Per-dimension size (half of head_dim for 2D, third for 3D)
        max_len : Maximum sequence length to precompute for
        base    : RoPE base (higher base = slower rotation = longer context range)

    Returns:
        cos, sin : each (max_len, dim)
    """
    # theta_i = 1 / 10000^(2i/dim) for i = 0 .. dim//2 - 1
    i     = torch.arange(0, dim // 2, dtype=torch.float32)
    theta = 1.0 / (base ** (2 * i / dim))             # (dim//2,)

    pos   = torch.arange(max_len, dtype=torch.float32) # (max_len,)

    # Outer product: position × frequency
    # (max_len, dim//2)
    angles = torch.outer(pos, theta)

    # Repeat so we can apply rotate_half without splitting
    # (max_len, dim//2) → (max_len, dim)
    angles = torch.cat([angles, angles], dim=-1)
    return angles.cos(), angles.sin()


def compute_2d_mrope_freqs(
    grid_h: int, grid_w: int, head_dim: int, base: float = 10000.0
) -> tuple:
    """
    Computes 2D mRoPE cos/sin for all patches in a spatial grid.

    The head_dim is split equally:
    - First half  → encodes row position (height_id)
    - Second half → encodes column position (width_id)

    Args:
        grid_h   : Number of patch rows
        grid_w   : Number of patch columns
        head_dim : Full per-head dimension

    Returns:
        cos, sin : each (num_patches, head_dim)  —  num_patches = grid_h × grid_w
    """
    half = head_dim // 2

    # Precompute 1D cos/sin tables for the half-dimension
    cos_1d, sin_1d = precompute_1d_freqs(half, max_len=max(grid_h, grid_w) + 1, base=base)

    # Position IDs for every patch in the grid
    # Row position: patch (h, w) has row_id = h
    # (num_patches,)
    row_ids = torch.arange(grid_h).repeat_interleave(grid_w)
    # Column position: patch (h, w) has col_id = w
    # (num_patches,)
    col_ids = torch.arange(grid_w).repeat(grid_h)

    # Gather the precomputed values at the patch positions
    # (num_patches, half)
    cos_h = cos_1d[row_ids]
    sin_h = sin_1d[row_ids]
    cos_w = cos_1d[col_ids]
    sin_w = sin_1d[col_ids]

    # Concatenate along head_dim: [row_encoding | col_encoding]
    # (num_patches, head_dim)
    cos_2d = torch.cat([cos_h, cos_w], dim=-1)
    sin_2d = torch.cat([sin_h, sin_w], dim=-1)

    return cos_2d, sin_2d


def compute_3d_mrope_freqs(
    num_frames: int, grid_h: int, grid_w: int, head_dim: int, base: float = 10000.0
) -> tuple:
    """
    Computes 3D mRoPE for video: each token has position (time, height, width).

    head_dim is split into thirds:
    - Dim [0, d/3)   → time position
    - Dim [d/3, 2d/3) → row position
    - Dim [2d/3, d)  → column position

    Args:
        num_frames : Number of video frames
        grid_h     : Patch rows per frame
        grid_w     : Patch columns per frame
        head_dim   : Full per-head dimension (should be divisible by 3 for clean split)

    Returns:
        cos, sin : each (total_patches, head_dim)
                   where total_patches = num_frames × grid_h × grid_w
    """
    third = head_dim // 3
    num_patches_per_frame = grid_h * grid_w

    cos_1d, sin_1d = precompute_1d_freqs(third, max_len=max(num_frames, grid_h, grid_w) + 1, base=base)

    total_patches = num_frames * num_patches_per_frame

    # Time position: all patches in frame t share the same time_id = t
    # (total_patches,)
    time_ids = torch.arange(num_frames).repeat_interleave(num_patches_per_frame)
    # Spatial positions within each frame
    row_ids  = torch.arange(grid_h).repeat_interleave(grid_w).repeat(num_frames)  # (total_patches,)
    col_ids  = torch.arange(grid_w).repeat(grid_h).repeat(num_frames)             # (total_patches,)

    # (total_patches, third) for each dimension
    cos_t = cos_1d[time_ids];  sin_t = sin_1d[time_ids]
    cos_h = cos_1d[row_ids];   sin_h = sin_1d[row_ids]
    cos_w = cos_1d[col_ids];   sin_w = sin_1d[col_ids]

    # Concatenate: [time | height | width] along head_dim
    # (total_patches, head_dim)
    cos_3d = torch.cat([cos_t, cos_h, cos_w], dim=-1)
    sin_3d = torch.cat([sin_t, sin_h, sin_w], dim=-1)

    return cos_3d, sin_3d


def apply_mrope(
    q: torch.Tensor, k: torch.Tensor,
    cos: torch.Tensor, sin: torch.Tensor,
) -> tuple:
    """
    Applies mRoPE to query and key tensors.

    The cos/sin encode the full position (height + width or time + height + width)
    and are applied uniformly to all heads.

    Args:
        q, k  : (batch_num, num_heads, seq_len, head_dim)
        cos   : (seq_len, head_dim)  mRoPE cosines
        sin   : (seq_len, head_dim)  mRoPE sines

    Returns:
        q_rot, k_rot : (batch_num, num_heads, seq_len, head_dim)
    """
    # Broadcast cos/sin from (seq_len, head_dim) to (1, 1, seq_len, head_dim)
    cos = cos.unsqueeze(0).unsqueeze(0).to(q.device)
    sin = sin.unsqueeze(0).unsqueeze(0).to(q.device)

    # RoPE rotation: x' = x*cos + rotate_half(x)*sin
    # (batch_num, num_heads, seq_len, head_dim)
    q_rot = q * cos + rotate_half(q) * sin
    k_rot = k * cos + rotate_half(k) * sin

    return q_rot, k_rot


# ── Visualise 2D mRoPE position patterns ──────────────────────────────────────
grid_h, grid_w = 7, 10  # example: 98×140 image with patch=14
head_dim = 64

cos_2d, sin_2d = compute_2d_mrope_freqs(grid_h, grid_w, head_dim)
# (num_patches=70, head_dim=64)

print(f"2D mRoPE cos shape : {cos_2d.shape}")  # (70, 64)
print(f"3D mRoPE for video:")
cos_3d, sin_3d = compute_3d_mrope_freqs(num_frames=8, grid_h=7, grid_w=10, head_dim=96)
print(f"  cos shape : {cos_3d.shape}")          # (560, 96) = 8*7*10 patches

# Show the spatial structure of the first two mRoPE dimensions
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, dim_slice, title in [
    (axes[0], slice(0, head_dim//2),  "Height encoding  (dim 0 … d/2)"),
    (axes[1], slice(head_dim//2, None),"Width encoding   (dim d/2 … d)"),
    (axes[2], slice(0, 4),             "First 4 dims (position fingerprint)"),
]:
    data = cos_2d[:, dim_slice].detach().numpy()
    im = ax.imshow(data.reshape(grid_h, grid_w, -1).mean(-1), cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("patch col"); ax.set_ylabel("patch row")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("mRoPE cosine values per patch position  (7×10 grid)")
plt.tight_layout(); plt.show()

## 7.3 Window Attention + Periodic Global Attention

**The efficiency problem with full attention**

Full self-attention is $O(N^2)$ in the number of tokens.
A 4K image at patch=14 produces ~34 000 tokens — full attention is infeasible.

**Window attention** (borrowed from Swin Transformer, applied here to plain ViT):
- Partition the patch grid into non-overlapping windows of size $W_s \times W_s$
- Run self-attention only within each window: $O(N \cdot W_s^2)$ — linear in $N$
- Windows are independent — no cross-window communication

**The connectivity problem and the fix**

Windows with no cross-window communication break long-range understanding.
The Qwen2.5-VL fix:
> Apply **full global attention every K layers** (typically K=4).

This is simpler than Swin's cyclic shift (no masking needed), and aligns better
with LLM attention which is always global.

In [ ]:
def window_partition(
    x: torch.Tensor, window_size: int
) -> tuple:
    """
    Partitions a 2D spatial feature map into non-overlapping windows.

    Args:
        x           : (batch_num, grid_h, grid_w, embed_dim) — spatial patch features
        window_size : Size of each square window (in patches)

    Returns:
        windows     : (num_windows, window_size^2, embed_dim)
        (batch_num, grid_h, grid_w) for reversal
    """
    batch_num, grid_h, grid_w, embed_dim = x.shape

    # Pad if grid is not divisible by window_size
    pad_h = (window_size - grid_h % window_size) % window_size
    pad_w = (window_size - grid_w % window_size) % window_size
    if pad_h > 0 or pad_w > 0:
        x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))  # pad last two spatial dims

    _, H_pad, W_pad, _ = x.shape
    n_win_h = H_pad // window_size
    n_win_w = W_pad // window_size

    # Reshape and permute to extract windows
    # (batch_num, n_win_h, window_size, n_win_w, window_size, embed_dim)
    x = x.view(batch_num, n_win_h, window_size, n_win_w, window_size, embed_dim)
    # (batch_num, n_win_h, n_win_w, window_size, window_size, embed_dim)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    # (num_windows * batch_num, window_size^2, embed_dim)
    windows = x.view(-1, window_size * window_size, embed_dim)

    return windows, (batch_num, grid_h, grid_w, H_pad, W_pad)


def window_reverse(
    windows: torch.Tensor,
    window_size: int,
    meta: tuple,
) -> torch.Tensor:
    """
    Reverses window_partition, removing any padding.

    Args:
        windows     : (num_windows * batch_num, window_size^2, embed_dim)
        window_size : Same as used in window_partition
        meta        : (batch_num, orig_h, orig_w, H_pad, W_pad)

    Returns:
        x : (batch_num, orig_h, orig_w, embed_dim)
    """
    batch_num, grid_h, grid_w, H_pad, W_pad = meta
    n_win_h = H_pad // window_size
    n_win_w = W_pad // window_size
    embed_dim = windows.shape[-1]

    # (num_windows * batch_num, window_size^2, C) → (batch_num, n_win_h, n_win_w, ws, ws, C)
    x = windows.view(batch_num, n_win_h, n_win_w, window_size, window_size, embed_dim)
    # (batch_num, n_win_h, ws, n_win_w, ws, C) → (batch_num, H_pad, W_pad, C)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(batch_num, H_pad, W_pad, embed_dim)

    # Remove padding
    # (batch_num, grid_h, grid_w, embed_dim)
    return x[:, :grid_h, :grid_w, :].contiguous()


class WindowedMHSA(nn.Module):
    """
    Multi-head self-attention with optional windowing + mRoPE.

    When window_size is None or global_attn=True: runs full global attention.
    Otherwise: partitions into windows, attends within each, reassembles.

    Args:
        embed_dim   : Token embedding dimension
        num_heads   : Number of attention heads
        window_size : Local window size in patches (None = always global)
    """

    def __init__(self, embed_dim: int, num_heads: int, window_size: int = 4) -> None:
        super().__init__()
        self.num_heads  = num_heads
        self.head_dim   = embed_dim // num_heads
        self.scale      = self.head_dim ** -0.5
        self.window_size = window_size

        # Fused QKV projection (same as standard MHSA)
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, 3*embed_dim)
        self.qkv      = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def _attend(
        self, x: torch.Tensor, cos: torch.Tensor = None, sin: torch.Tensor = None
    ) -> torch.Tensor:
        """Core MHSA with optional mRoPE. x: (B, N, C) → (B, N, C)."""
        B, N, C = x.shape

        # (B, N, 3*C) → 3 × (B, num_heads, N, head_dim)
        qkv = self.qkv(x).view(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        Q, K, V = qkv.unbind(0)  # each (B, num_heads, N, head_dim)

        # Apply 2D mRoPE if position frequencies are provided
        if cos is not None and sin is not None:
            Q, K = apply_mrope(Q, K, cos, sin)

        # Scaled dot-product attention
        # (B, num_heads, N, N)
        attn = (Q @ K.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)

        # (B, num_heads, N, head_dim) → (B, N, embed_dim)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, N, C)
        return self.out_proj(out)

    def forward(
        self,
        x: torch.Tensor,
        grid_h: int,
        grid_w: int,
        cos: torch.Tensor = None,
        sin: torch.Tensor = None,
        global_attn: bool = False,
    ) -> torch.Tensor:
        """
        Args:
            x           : (batch_num, num_patches, embed_dim) patch tokens
            grid_h      : Patch grid height
            grid_w      : Patch grid width
            cos, sin    : (num_patches, head_dim) mRoPE frequencies
            global_attn : Force full global attention regardless of window_size

        Returns:
            out : (batch_num, num_patches, embed_dim)
        """
        if global_attn or self.window_size is None:
            # Full global attention — O(N²) but needed periodically
            return self._attend(x, cos, sin)

        batch_num, _, embed_dim = x.shape

        # Reshape to spatial grid for windowing
        # (batch_num, num_patches, embed_dim) → (batch_num, grid_h, grid_w, embed_dim)
        x_spatial = x.view(batch_num, grid_h, grid_w, embed_dim)

        # Partition into windows
        # (num_windows * batch_num, window_size^2, embed_dim)
        windows, meta = window_partition(x_spatial, self.window_size)

        # mRoPE: we need per-window position frequencies
        # For windowed attention, crop the relevant patch positions per window
        # (Simplified: use full-sequence cos/sin — only positions within each window are attended)
        win_cos, win_sin = (cos, sin) if cos is not None else (None, None)

        # Run attention within each window
        # (num_windows * batch_num, window_size^2, embed_dim)
        win_out = self._attend(windows, win_cos[:self.window_size**2] if win_cos is not None else None,
                                         win_sin[:self.window_size**2] if win_sin is not None else None)

        # Reassemble spatial feature map and flatten
        # (num_windows * batch_num, ws^2, C) → (batch_num, grid_h, grid_w, embed_dim)
        out_spatial = window_reverse(win_out, self.window_size, meta)

        # (batch_num, grid_h, grid_w, embed_dim) → (batch_num, num_patches, embed_dim)
        return out_spatial.view(batch_num, -1, embed_dim)


# ── Shape verification ─────────────────────────────────────────────────────────
wmhsa = WindowedMHSA(embed_dim=128, num_heads=4, window_size=4)
x_win = torch.zeros(2, 7 * 10, 128)  # 2 images, 70 patches (7×10 grid)
cos_v, sin_v = compute_2d_mrope_freqs(7, 10, wmhsa.head_dim)

with torch.no_grad():
    out_win  = wmhsa(x_win, grid_h=7, grid_w=10, cos=cos_v, sin=sin_v, global_attn=False)
    out_glob = wmhsa(x_win, grid_h=7, grid_w=10, cos=cos_v, sin=sin_v, global_attn=True)

print(f"Window attention output : {out_win.shape}")   # (2, 70, 128)
print(f"Global attention output : {out_glob.shape}")  # (2, 70, 128)

## 7.4 SwiGLU + RMSNorm — LLM-Aligned FFN and Normalisation

**Why replace GELU + LayerNorm?**

Alignment with the LLM that processes visual tokens:
- The LLM (e.g., Qwen2.5) uses SwiGLU + RMSNorm internally
- If the visual encoder uses the same components, features are in the same
  "computational dialect" — easier for cross-modal fusion via cross-attention

**RMSNorm vs LayerNorm**
$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_i x_i^2 + \epsilon}} \cdot \gamma$$

Omits the mean-subtraction step of LayerNorm. This is ~15% faster and
performs identically in practice for large models.

**SwiGLU** (Shazeer, 2020)
$$\text{SwiGLU}(x, W, V, W_2) = (\text{Swish}(xW) \otimes xV) W_2$$

The Swish gate allows the MLP to *selectively activate* dimensions,
providing better gradient flow than a plain ReLU or GELU MLP.

In [ ]:
class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalisation.

    Equivalent to LayerNorm but without mean subtraction.
    Used in Qwen2.5-VL, LLaMA, Mistral, and most modern LLMs.

    Args:
        dim : Feature dimension to normalise over
        eps : Stability constant
    """

    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))  # learnable scale γ
        self.eps    = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (..., dim)
        Returns:
            normalised x, same shape
        """
        # Compute RMS over the last dimension
        # (..., dim) → (..., 1)
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        # Scale by learned weight: (..., dim)
        return x * rms * self.weight


class SwiGLU(nn.Module):
    """
    SwiGLU Feed-Forward Network.

    Architecture: x → [gate_proj | up_proj] → Swish(gate) ⊗ up → down_proj

    The gate mechanism (Swish activation controlling the up-projection)
    provides a learned, input-dependent non-linearity — more expressive
    than a standard two-layer MLP.

    Using 2/3 × (4 × embed_dim) for hidden_dim follows the convention that
    keeps parameter count equal to a standard 4× FFN.

    Args:
        embed_dim  : Input/output dimension
        hidden_dim : Inner gate dimension (typically 8/3 × embed_dim for 4× expansion)
    """

    def __init__(self, embed_dim: int, hidden_dim: int = None) -> None:
        super().__init__()
        if hidden_dim is None:
            # Standard convention: 2/3 * 4 * embed_dim to match 4× FFN parameter count
            hidden_dim = int(2 / 3 * 4 * embed_dim)

        # Gate and content projections — no bias (LLM convention)
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, hidden_dim)
        self.gate_proj = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.up_proj   = nn.Linear(embed_dim, hidden_dim, bias=False)
        # Contraction back to embed_dim
        # (batch_num, seq_len, hidden_dim) → (batch_num, seq_len, embed_dim)
        self.down_proj = nn.Linear(hidden_dim, embed_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (batch_num, seq_len, embed_dim)
        Returns:
            out : (batch_num, seq_len, embed_dim)
        """
        # Swish(gate) ⊗ up — element-wise gating
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, hidden_dim)
        gate = F.silu(self.gate_proj(x))   # Swish ≡ SiLU: x * sigmoid(x)
        up   = self.up_proj(x)

        # Element-wise gating: hidden_dim → hidden_dim
        gated = gate * up

        # Contract back
        # (batch_num, seq_len, hidden_dim) → (batch_num, seq_len, embed_dim)
        return self.down_proj(gated)


# ── Compare LayerNorm vs RMSNorm output ───────────────────────────────────────
x_norm_test = torch.randn(4, 16, 128)
ln = nn.LayerNorm(128)
rms = RMSNorm(128)

with torch.no_grad():
    ln_out  = ln(x_norm_test)
    rms_out = rms(x_norm_test)

print(f"LayerNorm output mean : {ln_out.mean():.5f}  std : {ln_out.std():.5f}")
print(f"RMSNorm   output mean : {rms_out.mean():.5f}  std : {rms_out.std():.5f}")
print(f"(RMSNorm does not subtract mean — the non-zero mean is expected)")

# Compare SwiGLU vs standard GELU MLP parameter count
swiglu = SwiGLU(128)
gelu_mlp = nn.Sequential(nn.Linear(128, 512), nn.GELU(), nn.Linear(512, 128))
print(f"\nSwiGLU params     : {sum(p.numel() for p in swiglu.parameters()):,}")
print(f"Standard MLP params: {sum(p.numel() for p in gelu_mlp.parameters()):,}")

## 7.5 Full Qwen-Style ViT Block and Encoder

Assembling all the components into a production-ready ViT block:

```
x_in
 ├─ RMSNorm → WindowedMHSA(mRoPE) → + x_in
 └─ RMSNorm → SwiGLU              → + x_in
x_out
```

**Full attention period**: every `full_attn_every` layers use global attention
instead of window attention to allow long-range communication.

In [ ]:
class QwenViTBlock(nn.Module):
    """
    Single transformer block matching Qwen2.5-VL's visual encoder design.

    Differences from standard ViT block:
    - RMSNorm instead of LayerNorm
    - SwiGLU instead of GELU MLP
    - mRoPE applied in attention (no absolute pos_embed needed)
    - Window attention by default; global attention when flagged
    """

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        window_size: int = 4,
        mlp_ratio: float = 8 / 3,
    ) -> None:
        super().__init__()
        self.norm1 = RMSNorm(embed_dim)
        self.attn  = WindowedMHSA(embed_dim, num_heads, window_size)
        self.norm2 = RMSNorm(embed_dim)
        self.ffn   = SwiGLU(embed_dim, hidden_dim=int(embed_dim * mlp_ratio))

    def forward(
        self,
        x: torch.Tensor,
        grid_h: int,
        grid_w: int,
        cos: torch.Tensor,
        sin: torch.Tensor,
        global_attn: bool = False,
    ) -> torch.Tensor:
        """
        Args:
            x           : (batch_num, num_patches, embed_dim)
            grid_h/w    : Spatial grid dimensions
            cos, sin    : (num_patches, head_dim) mRoPE frequencies
            global_attn : Use full attention for this layer

        Returns:
            out : (batch_num, num_patches, embed_dim)
        """
        # Pre-norm attention with residual
        # (batch_num, num_patches, embed_dim) → same
        x = x + self.attn(self.norm1(x), grid_h, grid_w, cos, sin, global_attn)

        # Pre-norm SwiGLU FFN with residual
        # (batch_num, num_patches, embed_dim) → same
        x = x + self.ffn(self.norm2(x))

        return x


class QwenViT(nn.Module):
    """
    Full Qwen2.5-VL style visual encoder.

    Key design decisions vs standard ViT:
    - No learned absolute positional embedding (mRoPE handles position)
    - No CLS token (VLMs use all patch tokens for cross-attention)
    - Window attention + periodic global attention for efficiency
    - RMSNorm + SwiGLU for LLM alignment
    """

    def __init__(
        self,
        in_channels: int   = 3,
        patch_size: int    = 14,
        embed_dim: int     = 512,
        num_heads: int     = 8,
        num_layers: int    = 12,
        window_size: int   = 4,
        full_attn_every: int = 4,   # use global attention every K layers
    ) -> None:
        super().__init__()
        self.patch_size     = patch_size
        self.full_attn_every = full_attn_every

        # Linear patch embedding (no positional embedding — mRoPE handles it)
        # (batch_num, in_channels, H, W) → (batch_num, num_patches, embed_dim)
        self.patch_embed = nn.Conv2d(in_channels, embed_dim,
                                     kernel_size=patch_size, stride=patch_size)

        self.blocks = nn.ModuleList([
            QwenViTBlock(embed_dim, num_heads, window_size)
            for _ in range(num_layers)
        ])

        self.norm = RMSNorm(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (batch_num, in_channels, H, W)  — H, W must be multiples of patch_size

        Returns:
            tokens : (batch_num, num_patches, embed_dim)  — all visual tokens
        """
        batch_num, _, H, W = x.shape
        p       = self.patch_size
        grid_h  = H // p
        grid_w  = W // p

        # Patch embedding — no CLS, no absolute pos embed
        # (batch_num, in_channels, H, W) → (batch_num, embed_dim, grid_h, grid_w)
        x = self.patch_embed(x)
        # (batch_num, embed_dim, grid_h, grid_w) → (batch_num, num_patches, embed_dim)
        x = x.flatten(2).transpose(1, 2)

        # Precompute 2D mRoPE frequencies for this spatial grid
        head_dim = self.blocks[0].attn.head_dim
        # (num_patches, head_dim)
        cos, sin = compute_2d_mrope_freqs(grid_h, grid_w, head_dim)

        for layer_idx, block in enumerate(self.blocks):
            # Every full_attn_every layers: use global attention for cross-window communication
            use_global = (layer_idx % self.full_attn_every == self.full_attn_every - 1)
            x = block(x, grid_h, grid_w, cos, sin, global_attn=use_global)

        # Final RMSNorm — no CLS extraction, return all patch tokens
        # (batch_num, num_patches, embed_dim)
        return self.norm(x)


# ── Shape dry-run ──────────────────────────────────────────────────────────────
torch.manual_seed(0)
qwen_vit = QwenViT(in_channels=3, patch_size=14, embed_dim=256,
                   num_heads=4, num_layers=8, window_size=4, full_attn_every=4)

print(f"QwenViT parameters : {sum(p.numel() for p in qwen_vit.parameters()):,}")

test_cases = [
    (224, 224),   # standard square
    (224, 336),   # portrait  (16:24 aspect ratio)
    (336, 224),   # landscape (24:16)
    (112, 448),   # extreme wide
]
with torch.no_grad():
    for H, W in test_cases:
        x_q   = torch.zeros(1, 3, H, W)
        out_q = qwen_vit(x_q)
        n_tok = (H // 14) * (W // 14)
        print(f"  Input {H}×{W:>4} → {n_tok:>4} tokens → output {out_q.shape}")

## 7.6 DINOv2 Features for Multimodal — Why Self-Supervised ViTs Work Better

**The supervised ViT's hidden bias**

A ViT trained with cross-entropy classification learns features optimised for
*distinguishing* ImageNet categories. These features have a critical flaw
for multimodal use: they discard visual information not needed for the 1000
ImageNet classes.

For a VLM answering *"what is written on the sign in this image?"*,
the relevant features (text texture, colour gradients) are exactly what
an ImageNet classifier ignores.

**Why DINOv2 features are better for VLMs**

| Property | Supervised ViT | DINOv2 |
|---|---|---|
| Training signal | Class labels | Structure of data itself |
| Feature specificity | Discriminative (class-separating) | Dense and semantic |
| Spatial features | Pooled away (CLS-only) | Patch-level, spatially rich |
| Texture sensitivity | Low (spurious feature) | High (spatial detail preserved) |
| OCR / document understanding | Poor | Strong |

DINOv2's self-supervised objective (student-teacher distillation) forces
the network to preserve *all* visually meaningful structure — it cannot
decide to ignore texture because no task tells it texture is irrelevant.

In [ ]:
# The key practical consequence: DINOv2 patch tokens contain rich dense features
# that work directly as input to an LLM's cross-attention — no projection needed
# in many small-scale experiments.

# Demonstration: compare CLS-token vs patch-token specificity

# Load both models
dino_v2   = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14", pretrained=True).eval()
resnet_cls = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).eval()

# Hook to extract intermediate features from ResNet
_rn_feats = {}
def _rn_hook(m, i, o): _rn_feats["out"] = o.detach()
resnet_cls.layer4.register_forward_hook(_rn_hook)

# Sample images: two different dogs (same class) and a cat (different class)
from torchvision.datasets import CIFAR10
import torchvision.transforms as T

eval_tf = T.Compose([T.Resize(224), T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
cifar   = CIFAR10("./data", train=False, download=True, transform=eval_tf)

# Pick a dog (class 5) and cat (class 3) sample
dog_idx  = [i for i, (_, y) in enumerate(cifar) if y == 5][:2]
cat_idx  = [i for i, (_, y) in enumerate(cifar) if y == 3][:1]

imgs = torch.stack([cifar[i][0] for i in dog_idx + cat_idx])  # (3, 3, 224, 224)

with torch.no_grad():
    # DINOv2 CLS features
    dino_out  = dino_v2(imgs)                        # (3, 768)  CLS token
    # DINOv2 patch features
    dino_dict = dino_v2.forward_features(imgs)
    dino_patches = dino_dict["x_norm_patchtokens"]   # (3, num_patches, 768)
    # ResNet spatial features → global avg pool
    _ = resnet_cls(imgs)
    rn_feats = _rn_feats["out"]                      # (3, 2048, 7, 7)
    rn_cls   = rn_feats.mean(dim=(2,3))              # (3, 2048)

# Cosine similarity between dog1 and dog2 vs dog1 and cat
def cos_sim(a, b): return F.cosine_similarity(a, b, dim=-1).item()

print("Feature similarity  (higher = more similar):")
print(f"  DINOv2 CLS    — dog vs dog: {cos_sim(dino_out[0], dino_out[1]):.3f}")
print(f"  DINOv2 CLS    — dog vs cat: {cos_sim(dino_out[0], dino_out[2]):.3f}")
print(f"  ResNet GAP    — dog vs dog: {cos_sim(rn_cls[0],   rn_cls[1]):.3f}")
print(f"  ResNet GAP    — dog vs cat: {cos_sim(rn_cls[0],   rn_cls[2]):.3f}")
print()
# Patch-level similarity captures local spatial features
patch_sim_dog = F.cosine_similarity(dino_patches[0], dino_patches[1], dim=-1).mean().item()
patch_sim_cat = F.cosine_similarity(dino_patches[0], dino_patches[2], dim=-1).mean().item()
print(f"  DINOv2 patches — dog vs dog: {patch_sim_dog:.3f}  (mean over all patches)")
print(f"  DINOv2 patches — dog vs cat: {patch_sim_cat:.3f}")

## 7.7 DeepStack — Multi-Layer Visual Feature Fusion (Qwen3-VL)

**Why use only the last ViT layer?**

Standard practice: pass only the final ViT layer's output to the LLM.

Problem: different layers capture different abstraction levels:
- Early layers: low-level texture, edges, colour
- Mid layers: object parts, spatial structure
- Late layers: semantic categories, global composition

For a task like *"describe the texture of the fabric"*, early layers are
more informative than the final layer. For *"what is the main subject?"*,
late layers dominate.

**DeepStack** (used in Qwen3-VL): aggregate features from multiple layers
using learned combination weights, giving the LLM access to multi-scale
visual information in a single token sequence.

In [ ]:
class DeepStack(nn.Module):
    """
    Fuses features from multiple ViT layers using learned weights.

    Registers forward hooks on every `select_every`-th block to capture
    intermediate representations, then combines them with a softmax-weighted
    sum followed by a learned projection.

    Args:
        blocks       : nn.ModuleList of ViT blocks (or QwenViTBlock)
        embed_dim    : Token embedding dimension
        select_every : Capture a layer every K blocks (default every 3rd)
    """

    def __init__(
        self,
        blocks: nn.ModuleList,
        embed_dim: int,
        select_every: int = 3,
    ) -> None:
        super().__init__()
        self.embed_dim   = embed_dim
        self.select_every = select_every

        # Which layer indices to capture
        self.selected = [
            i for i in range(len(blocks)) if (i + 1) % select_every == 0
        ]
        n_selected = len(self.selected)

        # One learnable weight per selected layer — softmaxed before combining
        # Initialise uniformly so training starts from equal weighting
        self.layer_weights = nn.Parameter(torch.zeros(n_selected))

        # Linear projection after fusion to re-scale the combined features
        # (batch_num, num_patches, embed_dim) → (batch_num, num_patches, embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim, bias=False)

        # Internal storage filled by forward hooks
        self._cached: list = []
        self._hooks: list  = []

    def register_hooks(self, blocks: nn.ModuleList) -> None:
        """Attaches forward hooks to the selected blocks."""
        def make_hook(idx):
            def hook(m, inp, out):
                if isinstance(out, torch.Tensor):
                    self._cached.append(out.detach())
            return hook

        for i in self.selected:
            self._hooks.append(blocks[i].register_forward_hook(make_hook(i)))

    def remove_hooks(self) -> None:
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

    def forward(self) -> torch.Tensor:
        """
        Called *after* the full ViT forward pass has populated self._cached.

        Returns:
            fused : (batch_num, num_patches, embed_dim) — multi-scale visual tokens
        """
        assert len(self._cached) == len(self.selected), (
            f"Expected {len(self.selected)} cached layers, got {len(self._cached)}"
        )

        # Normalised weights: one per selected layer, sums to 1
        # (n_selected,)
        weights = F.softmax(self.layer_weights, dim=0)

        # Weighted sum of all captured layer outputs
        # each (batch_num, num_patches, embed_dim) → (batch_num, num_patches, embed_dim)
        fused = sum(w * feat for w, feat in zip(weights, self._cached))

        # Clear cache for the next forward pass
        self._cached.clear()

        # Final projection
        # (batch_num, num_patches, embed_dim) → (batch_num, num_patches, embed_dim)
        return self.proj(fused)


class QwenViTWithDeepStack(QwenViT):
    """
    QwenViT augmented with DeepStack multi-layer feature fusion.

    After the standard forward pass, instead of returning only the final
    layer's output, returns the weighted combination of multiple layer outputs.
    """

    def __init__(self, select_every: int = 3, **kwargs) -> None:
        super().__init__(**kwargs)
        self.deepstack = DeepStack(self.blocks, kwargs["embed_dim"], select_every)
        self.deepstack.register_hooks(self.blocks)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_num, _, H, W = x.shape
        p = self.patch_size
        grid_h, grid_w = H // p, W // p

        # Standard patch embedding
        # (batch_num, 3, H, W) → (batch_num, num_patches, embed_dim)
        x = self.patch_embed(x).flatten(2).transpose(1, 2)

        # mRoPE frequencies for this grid
        head_dim = self.blocks[0].attn.head_dim
        cos, sin = compute_2d_mrope_freqs(grid_h, grid_w, head_dim)

        # Run all blocks — DeepStack hooks capture selected layers automatically
        for layer_idx, block in enumerate(self.blocks):
            use_global = (layer_idx % self.full_attn_every == self.full_attn_every - 1)
            x = block(x, grid_h, grid_w, cos, sin, global_attn=use_global)

        # Combine selected intermediate layers (the hooks have filled the cache)
        # (batch_num, num_patches, embed_dim)
        fused = self.deepstack()

        # Add final-layer residual so the model can also use last-layer features
        return self.norm(x + fused)


# ── Demo ───────────────────────────────────────────────────────────────────────
torch.manual_seed(0)
qwen_ds = QwenViTWithDeepStack(
    in_channels=3, patch_size=14, embed_dim=256,
    num_heads=4, num_layers=12, window_size=4,
    full_attn_every=4, select_every=3,
)

selected = qwen_ds.deepstack.selected
print(f"DeepStack captures layers : {selected}  ({len(selected)} layers / 12 total)")
print(f"Layer weights (before training, softmax) : {F.softmax(qwen_ds.deepstack.layer_weights.detach(), dim=0).tolist()}")

x_ds = torch.zeros(1, 3, 224, 224)
with torch.no_grad():
    out_ds = qwen_ds(x_ds)
print(f"\nInput  : {x_ds.shape}")
print(f"Output : {out_ds.shape}  ← all patch tokens, multi-layer fused")
print(f"Params : {sum(p.numel() for p in qwen_ds.parameters()):,}")

qwen_ds.deepstack.remove_hooks()

# Section 7 Summary — Modern Dynamic-Resolution ViT

**The evolution from Section 1 → Section 7**

```
Section 1:  ViT, fixed 28×28, learned pos_embed                ← understand the basics
Section 4.5: Bicubic interpolation of pos_embed at inference    ← enable variable resolution
Section 4.6: Rectangular inputs                                 ← aspect ratio awareness
Section 7:   Native aspect ratio + mRoPE + window attn + ...   ← production architecture
```

**Design decision rationale**

| Decision | Motivation |
|---|---|
| No absolute pos_embed | mRoPE handles position; no interpolation needed at inference |
| Window attn (local) | $O(N)$ for long sequences from high-res images |
| Global attn every K | Long-range context without $O(N^2)$ at every layer |
| RMSNorm | 15% faster; same performance; matches LLM internal norm |
| SwiGLU | More expressive gate; matches LLM FFN |
| No CLS token | VLMs consume all patch tokens via cross-attention |
| DeepStack | Multi-scale visual information in one token sequence |

**Key paper chain**
1. ViT (2020) — attention for images, fixed resolution
2. DINOv2 (2023) — SSL visual features for dense tasks
3. Swin (2021) — local window attention for visual transformers
4. Qwen2.5-VL (2024) — native resolution, mRoPE, window attention for VLMs
5. Qwen3-VL (2025) — DeepStack, deeper multi-layer fusion